# Actividad 8 — Generación de imágenes a partir de texto con Stable Diffusion (KerasCV)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ecamposv/nlp-vision/blob/main/semana-10/Actividad_8.ipynb)

Puedes ejecutar este notebook localmente (VS Code / Jupyter) o en **Google Colab** dando clic en el badge de arriba.

> ⚠️ **Este notebook es una BASE / plantilla.** Trae el andamiaje y ejemplos mínimos para arrancar, pero **tú debes completar** las secciones marcadas con `# TODO` y con ✍️ **Tu tarea**: el diseño del *prompt* refinado, las variantes, la tabla de evaluación y la justificación final.

## Modelos multimodales y generación texto→imagen

En esta práctica trabajamos con un modelo **generativo multimodal** de *texto a imagen* (**Stable Diffusion** vía **KerasCV**). El *prompt* (texto) funciona como una **guía semántica** que condiciona el resultado visual. En el ecosistema multimodal también existen modelos como **CLIP** (mide qué tan alineados están texto e imagen) y **BLIP** (describe imágenes con texto); al final hay un **reto opcional** que los usa para evaluar la coherencia de forma cuantitativa.

> 💡 **GPU recomendada.** La generación es mucho más rápida con GPU. En Colab: `Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU`. La **primera** generación descarga los pesos del modelo (~varios GB) y puede tardar.

## 1. Contexto de la práctica

**Campaña:** *"Ciudades sostenibles e inteligentes: tecnología para un futuro responsable"*.

| Elemento | Definición |
| --- | --- |
| **Problema** | Comunicar de forma visual y atractiva cómo la tecnología limpia puede transformar las ciudades y motivar prácticas sostenibles. |
| **Objetivo** | Generar imágenes que representen una ciudad moderna con tecnología limpia, áreas verdes, energía renovable y movilidad eléctrica, y **analizar cómo el diseño del *prompt*** influye en el resultado. |
| **Público objetivo** | Estudiantes universitarios. |
| **Concepto visual** | Ciudad inteligente y sostenible: paneles solares, aerogeneradores, transporte eléctrico, edificios con vegetación (*green architecture*), cielos limpios y personas conviviendo con la naturaleza. |

> ✍️ **Tu tarea (opcional):** Ajusta la tabla si quieres personalizar el concepto visual de tu campaña (por ejemplo, enfocarlo en movilidad, en energía o en arquitectura).

## 2. Instalación de librerías

La actividad pide **TensorFlow** y **KerasCV**. En **Google Colab** ya viene TensorFlow preinstalado, así que **no usamos `--upgrade`**: forzarlo actualiza TensorFlow a una versión que rompe paquetes del sistema (`tf-keras`, `tensorflow-text`, `protobuf`, `ydf`...) y produce conflictos de dependencias. En su lugar **fijamos versiones compatibles**:

```bash
# Colab (TensorFlow 2.20 preinstalado):
!pip install -q keras-cv "tensorflow==2.20.*"
```

La siguiente celda lo hace automáticamente en **Colab**. En un entorno **local**, ejecútala solo la primera vez.

> Si Colab muestra el aviso *"Restart runtime"* tras instalar, reinicia el entorno (**Entorno de ejecución → Reiniciar entorno de ejecución**) y vuelve a ejecutar desde aquí.

In [ ]:
# === Instalación de dependencias (Colab / primera ejecución local) ===
# En Colab NO usamos --upgrade para no actualizar TensorFlow y romper
# los paquetes preinstalados (tf-keras, tensorflow-text, protobuf, ydf...).
# Fijamos versiones compatibles con la TensorFlow 2.20 que trae Colab.
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q keras-cv "tensorflow==2.20.*"
    print("Setup de Colab completado.")
    print("Si Colab pide reiniciar el entorno, hazlo (Entorno de ejecución -> Reiniciar) y re-ejecuta desde aquí.")
else:
    # En local, instala una sola vez si no las tienes:
    #   pip install keras-cv "tensorflow>=2.16,<2.21" matplotlib
    print("Entorno local detectado. Asegúrate de tener: tensorflow, keras-cv, matplotlib.")

## 3. Importar librerías

In [ ]:
# Importar las librerías principales
import time
import numpy as np
import tensorflow as tf
import keras_cv
import matplotlib.pyplot as plt

print("TensorFlow:", tf.__version__)
print("KerasCV   :", keras_cv.__version__)

# ¿Hay GPU disponible?
gpus = tf.config.list_physical_devices("GPU")
print("GPU(s) detectada(s):", gpus if gpus else "ninguna (se usará CPU, será más lento)")

# (Opcional) Acelera la generación con precisión mixta en GPU (descomenta si tu GPU lo soporta):
# tf.keras.mixed_precision.set_global_policy("mixed_float16")

### Función auxiliar para visualizar

Definimos `mostrar_imagenes()` para graficar en una fila las imágenes que devuelve el modelo (un arreglo NumPy con forma `(n, alto, ancho, 3)`).

In [ ]:
def mostrar_imagenes(imagenes, titulos=None, figsize=(18, 6)):
    """Muestra en una fila las imágenes generadas por el modelo.

    imagenes : np.ndarray con forma (n, alto, ancho, 3), valores 0-255 (uint8).
    titulos  : lista opcional de títulos, uno por imagen.
    """
    n = len(imagenes)
    plt.figure(figsize=figsize)
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(imagenes[i])
        if titulos is not None:
            plt.title(titulos[i], fontsize=10)
        plt.axis("off")
    plt.tight_layout()
    plt.show()

## 4. Cargar el modelo generativo (Stable Diffusion)

Cargamos **Stable Diffusion** desde KerasCV. La primera vez descargará los pesos (puede tardar varios minutos). El objeto `modelo` expone el método `text_to_image(prompt, batch_size=...)`, que devuelve las imágenes generadas.

> 🛠️ **Si hay errores de compatibilidad de versiones** (TensorFlow / Keras 3 / KerasCV), fija versiones y **reinicia el entorno**, por ejemplo:
> `!pip install -q "keras-cv==0.9.0" "tensorflow==2.20.*"`

In [ ]:
# Stable Diffusion de KerasCV a 512x512 px.
# jit_compile=True acelera la inferencia (XLA) cuando hay GPU compatible.
modelo = keras_cv.models.StableDiffusion(
    img_width=512,
    img_height=512,
    jit_compile=True,
)
print("Modelo Stable Diffusion cargado. Listo para generar imágenes.")

## 5. Prompt base — imagen de referencia

Generamos una **primera imagen** con un *prompt* sencillo. Servirá como **referencia** para compararla con el *prompt* refinado y con las variantes.

Usamos un `SEED` fijo para que los resultados sean **reproducibles** y las comparaciones sean justas.

> 💬 **Sugerencia:** Stable Diffusion responde mejor a *prompts* en **inglés**.

In [ ]:
SEED = 42  # semilla fija para reproducibilidad

# Prompt base: descripción simple del concepto.
prompt_base = "a sustainable smart city"

inicio = time.time()
imagenes_base = modelo.text_to_image(prompt_base, batch_size=1, seed=SEED)
print(f"Generado en {time.time() - inicio:.1f} s")

mostrar_imagenes(imagenes_base, titulos=["Prompt base"], figsize=(6, 6))

## 6. Análisis del prompt base y refinamiento

Observa la imagen anterior y **analiza** la presencia (o ausencia) de elementos asociados con la sostenibilidad.

> ✍️ **Tu tarea — Análisis del prompt base**
> - ¿Aparecen elementos de sostenibilidad (paneles solares, áreas verdes, movilidad eléctrica, energía renovable)?
> - Si el resultado es **ambiguo** o genérico, explica la **limitación del prompt** (p. ej. es demasiado corto o poco específico).
>
> _Escribe aquí tu análisis:_ …

### Prompt refinado

Ahora **refina** el *prompt* añadiendo más detalle sobre los elementos visuales, el estilo, los materiales, la iluminación, etc. Abajo tienes un punto de partida: **mejóralo** para alinearlo con la campaña.

In [ ]:
# TODO: Refina este prompt. Añade detalle sobre elementos visuales concretos,
#       estilo, materiales, iluminación, composición, calidad, etc.
#       Ideas de "keywords": solar panels, wind turbines, electric buses,
#       green rooftops, bike lanes, clean sky, photorealistic, highly detailed...
prompt_refinado = (
    "a modern sustainable smart city with solar panels and wind turbines, "
    "green rooftops, electric buses and bike lanes, clean blue sky"
    # ✍️ agrega aquí más detalle...
)

imagenes_refinado = modelo.text_to_image(prompt_refinado, batch_size=1, seed=SEED)
mostrar_imagenes(imagenes_refinado, titulos=["Prompt refinado"], figsize=(6, 6))

### Comparación: prompt base vs. refinado

Los mostramos lado a lado para identificar las mejoras.

> ✍️ **Tu tarea:** Describe qué mejoró al refinar el *prompt* y cómo se alinea mejor con el objetivo de la campaña.

In [ ]:
# Base y refinado juntos para compararlos.
comparacion = np.concatenate([imagenes_base, imagenes_refinado], axis=0)
mostrar_imagenes(comparacion, titulos=["Base", "Refinado"], figsize=(10, 6))

## 7. Variantes del concepto (estilo, iluminación, perspectiva)

Genera **3 variantes** del concepto refinado modificando **un** aspecto en cada una:
- **Estilo** (p. ej. *watercolor*, *3D render*, *isometric*),
- **Iluminación** (p. ej. *golden hour*, *night, neon lights*),
- **Perspectiva** (p. ej. *aerial view*, *street level*).

La **Variante 1** ya está resuelta como ejemplo. **Completa** la 2 y la 3.

In [ ]:
# Cada variante parte del concepto refinado y cambia UN aspecto.
prompts_variantes = {
    # Variante 1 (ejemplo resuelto): cambio de ESTILO -> ilustración digital.
    "Variante 1 - estilo": prompt_refinado + ", digital art illustration, vibrant colors",

    # TODO Variante 2: cambia la ILUMINACIÓN (p. ej. "at golden hour", "at night, neon lights").
    "Variante 2 - iluminacion": prompt_refinado + "",  # ✍️ completa aquí

    # TODO Variante 3: cambia la PERSPECTIVA (p. ej. "aerial drone view", "street level view").
    "Variante 3 - perspectiva": prompt_refinado + "",  # ✍️ completa aquí
}

imagenes_variantes = []
titulos_variantes = []
for titulo, prompt in prompts_variantes.items():
    print("Generando:", titulo)
    img = modelo.text_to_image(prompt, batch_size=1, seed=SEED)
    imagenes_variantes.append(img[0])
    titulos_variantes.append(titulo)

mostrar_imagenes(np.array(imagenes_variantes), titulos=titulos_variantes, figsize=(18, 6))

## 8. Evaluación — tabla comparativa

Evalúa **cada** imagen según su **calidad visual**, su **coherencia con el prompt** y añade tus **observaciones**. Sustituye `Alta / Media / Baja` por tu valoración y escribe tus notas.

> ✍️ **Tu tarea:** Completa la tabla.

| Imagen generada | Calidad visual | Coherencia con el *prompt* | Observaciones |
| --- | --- | --- | --- |
| **Prompt base** | Alta / Media / Baja | Alta / Media / Baja | — |
| **Prompt refinado** | Alta / Media / Baja | Alta / Media / Baja | — |
| **Variante 1** | Alta / Media / Baja | Alta / Media / Baja | — |
| **Variante 2** | Alta / Media / Baja | Alta / Media / Baja | — |
| **Variante 3** | Alta / Media / Baja | Alta / Media / Baja | — |

## 9. Selección de la mejor imagen y justificación

Elige la **mejor imagen** y justifica tu elección con base en los **elementos visuales**, el **mensaje transmitido** y su **adecuación** al objetivo de la campaña.

> ✍️ **Tu tarea:** Escribe aquí tu justificación (mínimo un párrafo).

In [ ]:
# TODO: elige tu mejor imagen para volver a mostrarla y (opcional) guardarla.
# Opciones: imagenes_base[0], imagenes_refinado[0],
#           imagenes_variantes[0], imagenes_variantes[1], imagenes_variantes[2]
mejor = imagenes_refinado[0]

mostrar_imagenes(np.array([mejor]), titulos=["Mejor imagen (mi elección)"], figsize=(6, 6))

# (Opcional) Guardar la imagen elegida en disco:
# from PIL import Image
# Image.fromarray(mejor).save("mejor_imagen_campana.png")
# print("Imagen guardada como mejor_imagen_campana.png")

## 🎁 Reto opcional — Coherencia con CLIP / descripción con BLIP

La evaluación de la sección 8 es **cualitativa**. Como reto, mide la coherencia texto↔imagen de forma **cuantitativa** con **CLIP**, o genera una descripción automática de tu imagen con **BLIP** (ambos vía `transformers`).

> Es **opcional** y requiere: `!pip install transformers torch`

Ideas:
- **CLIP:** calcula la similitud entre tu *prompt* y cada imagen; mayor puntaje ≈ mayor coherencia.
- **BLIP:** genera un *caption* de la imagen y compáralo con lo que quisiste representar.

> ✍️ **Tu tarea (opcional):** Implementa la evaluación con CLIP y contrasta el ranking automático con tu tabla de la sección 8.

In [ ]:
# (OPCIONAL) Evaluación de coherencia con CLIP.
# Descomenta y ejecuta tras instalar:  transformers y torch
#
# from transformers import CLIPProcessor, CLIPModel
# import torch
#
# clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
# clip_proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
#
# def coherencia_clip(imagen_uint8, prompt):
#     """Puntaje de similitud imagen-texto (mayor = más coherente)."""
#     inputs = clip_proc(text=[prompt], images=imagen_uint8, return_tensors="pt", padding=True)
#     with torch.no_grad():
#         out = clip_model(**inputs)
#     return out.logits_per_image.item()
#
# print("Base    :", coherencia_clip(imagenes_base[0], prompt_base))
# print("Refinado:", coherencia_clip(imagenes_refinado[0], prompt_refinado))
print("Reto opcional: descomenta el código de arriba tras instalar 'transformers' y 'torch'.")

## ✅ Checklist de entrega

- [ ] Contexto de la práctica definido (sección 1).
- [ ] Modelo cargado y **prompt base** generado (secciones 4–5).
- [ ] Análisis del prompt base y **prompt refinado** justificado (sección 6).
- [ ] **3 variantes** generadas modificando estilo, iluminación o perspectiva (sección 7).
- [ ] **Tabla comparativa** completa (sección 8).
- [ ] **Mejor imagen** seleccionada y **justificada** (sección 9).
- [ ] (Opcional) Reto con CLIP / BLIP.